# Light Attenuation (KD490) — All Norway

Downloads KD490 (diffuse attenuation at 490 nm) from CMEMS for 2025,
covering all of Norway using two products:
- **Atlantic** (1 km): `cmems_obs-oc_atl_bgc-transp_my_l3-multi-1km_P1D`
- **Arctic** (4 km): `cmems_obs-oc_arc_bgc-transp_my_l3-multi-4km_P1D`

The bounding box is derived from the Norway DEM raster. Atlantic is preferred in the overlap zone.
Uses GDAL for all raster operations to keep memory usage low.
The photic zone is computed at DEM resolution (50 m).

In [ ]:
from pathlib import Path

import copernicusmarine as cm
import numpy as np
from osgeo import gdal
from rasterio.warp import transform_bounds

from mnk import light

gdal.UseExceptions()

out_dir = Path(".")

kd_year: str = "2024"  #: Year to extract kd490 values from cmems


## 1. Bounding box from Norway DEM

In [2]:
dem_url = "/vsicurl/https://storage.googleapis.com/niva-geodata/MarintNaturKart/features/feature_norge_dem50_depth_filled.tif"

ds = gdal.Open(dem_url)
gt = ds.GetGeoTransform()
dem_xsize, dem_ysize = ds.RasterXSize, ds.RasterYSize
dem_left = gt[0]
dem_top = gt[3]
dem_right = dem_left + gt[1] * dem_xsize
dem_bottom = dem_top + gt[5] * dem_ysize
dem_srs = ds.GetSpatialRef()
ds = None

lon_min, lat_min, lon_max, lat_max = transform_bounds(
    "EPSG:25833", "EPSG:4326", dem_left, dem_bottom, dem_right, dem_top,
)
lon_min, lat_min = round(lon_min - 0.5, 1), round(lat_min - 0.5, 1)
lon_max, lat_max = round(lon_max + 0.5, 1), round(lat_max + 0.5, 1)

print(f"Norway bbox (WGS84): lon [{lon_min}, {lon_max}], lat [{lat_min}, {lat_max}]")
print(f"DEM grid: {dem_xsize} x {dem_ysize} @ 50 m, bounds: [{dem_left}, {dem_bottom}, {dem_right}, {dem_top}]")

Norway bbox (WGS84): lon [-2.2, 32.8], lat [57.0, 72.3]
DEM grid: 24431 x 30735 @ 50 m, bounds: [-99600.0, 6426000.0, 1121950.0, 7962750.0]


## 2. Download KD490 from CMEMS (2025)

In [3]:
start_date = f"{kd_year}-04-01"
end_date = f"{kd_year}-10-31"
print(f"Time range: {start_date} to {end_date}")

OVERLAP_LAT = 62.0

ds_atl = cm.open_dataset(
    dataset_id="cmems_obs-oc_atl_bgc-transp_my_l3-multi-1km_P1D",
    variables=["KD490"],
    minimum_longitude=lon_min,
    maximum_longitude=lon_max,
    minimum_latitude=lat_min,
    maximum_latitude=min(lat_max, 66.0),
    start_datetime=start_date,
    end_datetime=end_date,
)
print(f"Atlantic shape: {ds_atl['KD490'].shape}")

Time range: 2025-04-01 to 2025-10-31


INFO - 2026-08-19T06:23:17Z - Selected dataset version: "202603"
INFO - 2026-08-19T06:23:17Z - Selected dataset part: "default"
WARNING - 2026-08-19T06:23:17Z - Some of your subset selection [57.0, 66.0] for the latitude dimension exceed the dataset coordinates [20.005207061767578, 65.99478912353516]
WARNING - 2026-08-19T06:23:17Z - Some of your subset selection [-2.2, 32.8] for the longitude dimension exceed the dataset coordinates [-45.99479293823242, 12.994793891906738]
WARNING - 2026-08-19T06:23:17Z - Some of your subset selection [57.0, 66.0] for the latitude dimension exceed the dataset coordinates [20.005207061767578, 65.99478912353516]
WARNING - 2026-08-19T06:23:17Z - Some of your subset selection [-2.2, 32.8] for the longitude dimension exceed the dataset coordinates [-45.99479293823242, 12.994793891906738]


Atlantic shape: (214, 864, 1459)


In [4]:
ds_arc = cm.open_dataset(
    dataset_id="cmems_obs-oc_arc_bgc-transp_my_l3-multi-4km_P1D",
    variables=["KD490"],
    minimum_longitude=lon_min,
    maximum_longitude=lon_max,
    minimum_latitude=OVERLAP_LAT,
    maximum_latitude=lat_max,
    start_datetime=start_date,
    end_datetime=end_date,
)
print(f"Arctic shape: {ds_arc['KD490'].shape}")

INFO - 2026-08-19T06:23:19Z - Selected dataset version: "202311"
INFO - 2026-08-19T06:23:19Z - Selected dataset part: "default"
WARNING - 2026-08-19T06:23:19Z - Some of your subset selection [62.0, 72.3] for the latitude dimension exceed the dataset coordinates [66.0, 90.00000000000091]
WARNING - 2026-08-19T06:23:19Z - Some of your subset selection [62.0, 72.3] for the latitude dimension exceed the dataset coordinates [66.0, 90.00000000000091]


Arctic shape: (214, 210, 389)


## 3. Compute temporal mean and mosaic

Prefer Atlantic (1 km) in overlap zone; fill gaps with Arctic (4 km interpolated to same grid).

In [ ]:
kd_merged, lons, lats = light.merge_kd490_datasets(
    ds_atl, ds_arc,
    lon_range=(lon_min, lon_max),
    lat_range=(lat_min, lat_max),
)
del ds_atl, ds_arc


## 4. Save merged KD490 as GeoTIFF (WGS84)

In [ ]:
kd_wgs84_path = str(out_dir / f"KD490_norge_{kd_year}_wgs84.tif")

light.save_wgs84_geotiff(kd_merged, lons, lats, kd_wgs84_path)
del kd_merged


Saved WGS84 mosaic: ../niva/KD490_norge_2025_wgs84.tif


## 5. Reproject to EPSG:25833 at 1000 m and save (before fill)

In [ ]:
kd_25833_path = str(out_dir / f"KD490_norge_{kd_year}_25833.tif")
output_bounds = [dem_left, dem_bottom, dem_right, dem_top]

light.reproject_to_25833(kd_wgs84_path, kd_25833_path, output_bounds)


KD490 25833 — shape: (1537, 1222), valid: 647227/1878214
Saved (unfilled): ../niva/KD490_norge_2025_25833.tif


## 6. Gap-fill KD490 using GDAL FillNodata

In [ ]:
kd_filled_path = str(out_dir / f"KD490_norge_{kd_year}_filled_25833.tif")

light.fill_kd490(kd_25833_path, kd_filled_path, output_bounds)


Fine fill (maxSearchDist=200)...
.100 - done.
0...10...20...30...40...50...60...70...80...90...Coarse fill (20x downsample, maxSearchDist=500)...
100 - done.
0...10...20...30...40...50...60...70...80...90...100 - done.
0...10...20...30...40...50...60...70...80...90..Filled — valid: 1875828/1878214
Saved: ../niva/KD490_norge_2025_filled_25833.tif


## 7. Compute photic zone at DEM resolution (50 m)

Resample filled KD490 (1 km) to the DEM grid (50 m) using GDAL, then apply:

$$z_{photic} = \frac{\ln(100)}{K_{d490}} \approx \frac{4.605}{K_{d490}}$$

A pixel is **photic** (1) if `|depth| < z_photic`, **aphotic** (0) otherwise.
Processed block-by-block to avoid loading the full DEM into memory.

In [ ]:
photic_path = str(out_dir / f"nisjedata-fotisk-sone-kd{kd_year}_norge_2026_25833.tif")

light.compute_photic_zone(
    dem_path=dem_url,
    kd_path=kd_filled_path,
    out_path=photic_path,
    output_bounds=output_bounds,
    dem_xsize=dem_xsize,
    dem_ysize=dem_ysize,
)


KD490 resampled to DEM grid: 24431 x 30735
